# EPFL 26-bus hybrid AC/DC microgrid

The low-voltage hybrid AC/DC microgrid of the EPFL laboratory (paper section IV-A, Fig. 4): an AC grid (0.4 kV) and a DC grid (0.8 kV) coupled by four converters, each connected through a cable (AC side B15-B18, DC side B19-B22).

| | |
|---|---|
| AC buses | 18 (17 lines, 0 transformers, 0 generators) |
| DC buses | 8, in 1 DC grid |
| Converters | 4: 3 × VdcQ (holds its DC voltage, fixed Q), 1 × PQ (fixed P and Q) |
| Case file | `validation/cases/microgrid.yaml` |

## What this notebook does

1. **Load** the case file into a pandapower net and classify the nodes. `LOSSY = True` adds the converter
   losses a + b·|I| + c·|I|² (coefficients of the paper, section IV-B).
2. **Build Y**: the AC admittance matrix (lines, transformer taps and shifts, shunts) and the DC conductance
   matrix.
3. **Solve** the AC grids, the DC grids and the converters together in one Newton–Raphson, and print the
   iterations, the largest mismatch and the run time.
4. **Results**: per node the voltage E = E_re + j·E_im (p.u.) and the injected power P, Q (p.u., positive into
   the grid); per converter its AC and DC power, losses and current.

## Try it

- Change a converter in the `converter` table of the case file, for example its `mode`, `p_mw`, `q_mvar` or
  `vdc_pu`, then *Run All*. Modes: PQ, VdcQ, PV, VdcV, GF (grid-forming) and droop; the columns each mode uses are
  listed at the top of the case file.
- Set `LOSSY = True` in the first cell to include converter losses.

## Grid topology

![Grid topology](figures/microgrid.png)

Bus numbers in the case file are the B-numbers of the figure. The converters are named `IC <DC bus>-<AC bus>`: IC 1 = `IC 22-18`, IC 2 = `IC 19-15`, IC 3 = `IC 21-17`, IC 4 = `IC 20-16`.

## Running this notebook

Use the kernel **Python 3.12 (unified-acdc-powerflow)** (VS Code: *Select Kernel* → *Jupyter Kernel…*). To set it
up once, in the repository folder:

```bash
python3.12 -m venv .venv
.venv/bin/pip install -e ".[dev]"
.venv/bin/python -m ipykernel install --user --name unified-acdc-powerflow --display-name "Python 3.12 (unified-acdc-powerflow)"
```


In [1]:
import logging, warnings
logging.getLogger("pandapower").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_rows", 2000)
from unified_acdc_powerflow import build_model, build_ybus, solve
from unified_acdc_powerflow.cases import load_case

# 1) load the case and classify the nodes
CASE = "microgrid"
LOSSY = False          # True: converter losses a + b|I| + c|I|^2 (coefficients of the paper)
net = load_case(CASE, lossy=LOSSY)   # pandapower net from validation/cases/microgrid.yaml
model = build_model(net)
print(f"{model.n_ac} AC nodes, {model.n_dc} DC nodes, {model.n_conv} converters")
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu", "p_dc_ref_mw", "droop_k"]]


18 AC nodes, 8 DC nodes, 4 converters


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu,p_dc_ref_mw,droop_k
0,IC 19-15,15,19,VdcQ,0.0000,0.01,1.00,0.0,0.0
1,IC 20-16,16,20,VdcQ,0.0031,0.01,1.02,0.0,0.0
2,IC 21-17,17,21,VdcQ,-0.0017,0.01,0.98,0.0,0.0
3,IC 22-18,18,22,PQ,-0.0031,0.01,1.00,0.0,0.0


In [2]:
# 2) build Y (AC: pandapower Ybus with taps and shunts; DC: conductance matrix G)
build_ybus(model)
print("Y (AC):", model.Y.shape, "nnz", model.Y.nnz)
print("G (DC):", model.G.shape, "nnz", model.G.nnz)


Y (AC): (18, 18) nnz 52
G (DC): (8, 8) nnz 22


In [3]:
# 3) Newton-Raphson
res = solve(model)
print(res.message, "| iterations:", res.iterations, "| max mismatch:", f"{res.max_mismatch:.2e}",
      "| time:", f"{res.solve_time_s * 1e3:.1f} ms")


converged | iterations: 5 | max mismatch: 4.45e-14 | time: 50.2 ms


In [4]:
# 4) results: per node E (real, imag) and injected P, Q in p.u.; then per converter
display(res.table())
res.converter_table()


,side,bus,name,type,E_re,E_im,P_pu,Q_pu
0,ac,1,1,slack,1.000000,0.000000,1.216743e+00,-4.611150e-01
1,ac,2,2,PQ,0.987919,-0.011814,2.824623e-14,1.370338e-14
2,ac,3,3,PQ,0.968742,-0.012405,-3.000000e-01,-1.828349e-17
3,ac,4,4,PQ,0.983685,-0.016963,-6.808705e-15,1.060479e-14
4,ac,5,5,PQ,0.977907,-0.016665,-3.750000e-01,-7.500000e-02
5,ac,6,6,PQ,0.982377,-0.035293,6.980211e-15,-2.507698e-16
6,ac,7,7,PQ,0.982006,-0.040534,-5.760157e-16,-1.395515e-14
7,ac,8,8,PQ,0.973716,-0.027367,1.282472e-14,2.517229e-14
8,ac,9,9,PQ,0.973040,-0.028249,-1.010000e-01,-1.818054e-15
9,ac,10,10,PQ,0.951119,-0.047866,0.000000e+00,-0.000000e+00


,vsc,name,mode,P_ac_pu,Q_ac_pu,P_dc_pu,loss_pu,I_pu
0,0,IC 19-15,VdcQ,-0.066082,0.1,0.066082,0.0,0.123146
1,1,IC 20-16,VdcQ,-0.240657,0.1,0.240657,0.0,0.280726
2,2,IC 21-17,VdcQ,0.325276,0.1,-0.325276,0.0,0.352345
3,3,IC 22-18,PQ,-0.031000,0.1,0.031000,0.0,0.106516
